# 02 - Same field, different deterministic samplers

Notebook `01` trained one velocity field. We now freeze that field and ask two
questions. First, how closely do Euler, Heun, and RK4 follow it? Second, how can
the same velocity prediction be read as an estimate of clean data?

Keeping the model and initial noise fixed makes both questions concrete. A
solver changes the numerical path. The denoiser conversion changes how we read
the model's prediction, not what the model learned.

**Time convention:** `t=0` is noise and `t=1` is data.


In [ ]:
import sys
from pathlib import Path

NOTES_DIR = Path.cwd()
if not (NOTES_DIR / "toy_notes.py").exists():
    NOTES_DIR = Path("notebooks/toy_data")
sys.path.insert(0, str(NOTES_DIR.resolve()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from toy_notes import *
from notebook_views import *

SEED = 2026
set_seed(SEED)
device = choose_device()
print("device:", device)


In [ ]:
model, losses, _ = load_or_train_model(device=device)


## 1. What does deterministic mean here?

The model defines the arrows; the solver decides how to follow them. A fair
solver comparison changes only the solver and step count. It keeps the model,
initial noise tensor, and evaluation reference fixed.

Deterministic is conditional on the initial state: repeating the same
`x_noise` repeats the trajectory, while different initial noise points can
still produce a diverse batch of endpoints.

We use a 256-step RK4 trajectory as a numerical reference. This does not remove
model error; it only makes the numerical integration error small.

**Before you run:** Euler uses one model evaluation per step, Heun uses two, and
RK4 uses four. At a fixed number of steps, which should follow a curved field
most accurately? At a fixed number of model evaluations, is the answer
necessarily the same?


In [ ]:
def euler_step(velocity, t, x, dt):
    return x + dt * velocity(t, x)

def heun_step(velocity, t, x, dt):
    slope_now = velocity(t, x)
    predicted = x + dt * slope_now
    slope_next = velocity(t + dt, predicted)
    return x + 0.5 * dt * (slope_now + slope_next)

def integrate_with_step(velocity, x0, steps, step_fn):
    x = x0.detach().clone()
    path = [x.cpu()]
    dt = 1.0 / steps
    for index in range(steps):
        t = torch.full((len(x),), index * dt, device=x.device)
        x = step_fn(velocity, t, x, dt).detach()
        path.append(x.cpu())
    return torch.stack(path)

STEP_COUNTS = [8, 16, 32, 64]


In [ ]:
set_seed(SEED + 21)
x0 = sample_base(1200, device=device)
target, _ = sample_labeled_mixture(1200, device=device)

velocity = base_velocity(model)
reference_path = integrate_ode(velocity, x0, steps=256, method="rk4")
paths = {}
for name, step_fn in {"euler": euler_step, "heun": heun_step}.items():
    for steps in STEP_COUNTS:
        paths[(name, steps)] = integrate_with_step(velocity, x0, steps, step_fn)
for steps in STEP_COUNTS:
    paths[("rk4", steps)] = integrate_ode(velocity, x0, steps=steps, method="rk4")

rows, saved_paths = solver_metrics_from_paths(paths, reference_path[-1], target)
solver_table = pd.DataFrame(rows)
display(solver_table.round(4))


In [ ]:
plot_trajectory_comparison(saved_paths, n_lines=22)
plt.show()
plot_solver_accuracy(solver_table)


The paired endpoint error is meaningful because every solver starts from the
same tensor. A more accurate solver follows the learned field more faithfully;
it cannot repair a poorly learned field.

**Change one thing:** Replace `STEP_COUNTS` with `[4, 8, 16]`. Rerun the solver
cells and find the smallest computation budget at which the generated modes
remain recognizable.

## 2. How can velocity be read as a clean estimate?

For one known training pair,

$$x_t=x_{noise}+t(x_{data}-x_{noise}),\qquad
u=x_{data}-x_{noise}.$$

Rearranging gives the exact identity

$$x_{data}=x_t+(1-t)u.$$

During generation the endpoint pair is unknown, so the network substitutes its
learned velocity for `u`:

$$D_\theta(x_t,t)=x_t+(1-t)v_\theta(x_t,t).$$

**Before you run:** At `t=0.35`, should this estimate recover each hidden clean
endpoint exactly, or average over several endpoints compatible with the same
noisy state?


In [ ]:
set_seed(SEED + 19)
visual_clean, visual_labels = sample_labeled_mixture(180, device=device)
visual_noise = sample_base(180, device=device)
visual_t_value = 0.35
visual_t = torch.full((180,), visual_t_value, device=device)
visual_xt = forward_noising(visual_clean, visual_t, visual_noise)
pair_velocity = visual_clean - visual_noise
with torch.no_grad():
    learned_velocity = model(visual_t, visual_xt)
    clean_estimate = visual_xt + (1 - visual_t[:, None]) * learned_velocity

plot_velocity_to_denoiser(
    visual_xt, pair_velocity, learned_velocity, clean_estimate, visual_clean,
    t_value=visual_t_value,
)


The exact pair velocity recovers its paired endpoint because both endpoints were
used to construct it. The network sees only `(x_t,t)`. With mean-squared-error
training, the population prediction is a conditional average,

$$v^*(x_t,t)=\mathbb E[u\mid x_t,t],$$

so the converted estimate is

$$D^*(x_t,t)=\mathbb E[x_{data}\mid x_t,t].$$

It is a posterior mean, not a reconstruction of one secretly known original.
The full derivation and the endpoint caveat are in `BACKGROUND.md`.


In [ ]:
set_seed(SEED + 20)
clean, _ = sample_labeled_mixture(450, device=device)
noise = sample_base(clean.shape[0], device=device)
plot_denoiser_times(model, clean, noise)


## 3. Keep two uses of noise separate

A **same-seed paired comparison** is an evaluation design: two deterministic
methods start from the exact same noise tensor, so endpoint differences can be
attributed to the changed sampling rule.

Notebook `03` introduces a steering signal formed by subtracting a full-data
denoiser from a target-class denoiser. We will call it the
**class-minus-full denoiser correction** and show the subtraction explicitly.
It is unrelated to reusing a seed.

Notebook `03` begins with a visible question: can a denoiser built only from a
mean and principal directions recover useful structure from noisy MNIST
images?
